# CORE-WESM pipeline notebook – Workshop clean cooking analysis – 27 February 2026


## Introduction ##

<img src="figures/others/KNeCS_cover.png" alt="KNeCS cover page" width="200" style="float:right;"/>

This notebook guides through a simple example analysis using CORE-WESM. The analysis uses CORE-WESM to explore potential county-level implications of the clean cooking targets of the Kenya National electric Cooking Strategy (KNeCS), and by extension the Kenya National Cooking Transition Strategy (KNCTS).

Working through the document will facilitate the completion of the two overarching tasks:
1. Run three scenarios from the KNeCS with CORE-WESM to explore their implications across counties.
2. Reflect on the results. What county-level characteristics that could shape pathways differently are not captured? How could these type of modelling insights help support integrated planning? 

#### How to use the notebook: ####

- The notebook is organised in different cells, i.e., blocks, which contain either a) normal text (like this cell itself) or b) Python code, as well as linked outputs from the code (for example, graphs).
- The example analysis requires working through all cells of the notebook from top to bottom (scrolling required):
  - **Text cells (white background)** – These provide context and explanations of the workflow conducted in the following code cells.
  - **Code cells (gray background if not editing)** – **Each of the cells needs to be run** to perform the respective (part of a) workflow step. A cell can be run by clicking anywhere on it, and then clicking the *Run this cell and advance* button (<img src="figures/others/run_button.png" alt="Run button" width="14"/>) at the top, or pressing *Shift+Enter*. Some cells require changes to adjust the analysis, e.g., choose certain scenarios or counties, but all can be run without making changes. If a code cell requires changes, this is highlighted in the adjacent text cell as <b style='color:red;'>[Change required]</b>.
  - **Code cells output** – The output of the code cells generally provides a log of the running of the code (e.g, any errors), and in case of the analysis cells, output graphs that can be considered.
- The setup also includes configuration files referred to throughout the notebook, but these do not need to be considered or changed itself for this example analysis.


## (0) Setting up the Workflow ##

This initial step – before the actual workflow – **sets up the programming context for the workflow**. It loads required libraries, sets up a logger, and loads the configuration file, which includes, for example, the filepaths for data files required throughout the workflow (the configuration file can be found <a href="config_files/pipeline_config.yaml">here</a>).

In [ ]:
# Load relevant Python libraries
import yaml
import logging
import warnings
import pandas as pd
import COREWESM_data_pipeline as dp
import COREWESM_run_pipeline as rp

# Suppress certain warnings for better readibility of logs
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

# Set up the logger
dp.setup_logger("INFO")
logger = logging.getLogger(__name__)


# Load the workflow configuration file
with open("config_files/pipeline_config.yaml") as stream:
    try:
        cfg = yaml.safe_load(stream)
    except yaml.YAMLError as exc:
        print(exc)

logger.info("Libraries and config file loaded")

## (1) Processing the national model ##

This first main workflow step **converts a datafile** of the national OSeMOSYS Kenya model **to a spreadsheet format** for easier processing.

In [ ]:
# Convert the datafile to spreadsheet file (this makes use of the filepaths set out in configuration file)
dp.convert_datafile(dataconfig_file = cfg["filepaths"]["data_config_file_NM"],
                    data_file = cfg["filepaths"]["national_data_file"],
                    output_file = cfg["filepaths"]["national_spreadsheet_file"])

## (2) Downscaling the national model ##

The second step of the workflow uses the converted national model in spreadsheet format and downscales chosen sectors to the county level. Currently, this includes the residential, services, and agricultural sectors – with the residential sector being the key sector for this analysis.

In [ ]:
# Downscale to a county-resolved model (this again makes use of the filepaths and parameters set out in the configuration file to define how the downscaling is to be performed)
dp.downscale(input_file = cfg["filepaths"]["national_spreadsheet_file"],
              dataconfig_file = cfg["filepaths"]["data_config_file_NM"],
              tech_sector_mapping = cfg["downscaling"]["tech_to_sector_mapping_file"],
              comm_sector_mapping = cfg["downscaling"]["comm_to_sector_mapping_file"],
              list_counties = cfg["downscaling"]["list_counties_file"],
              pop_file = cfg["downscaling"]["pop_file"],
              pop_file_ruur = cfg["downscaling"]["pop_file_ruur"],
              gdp_file = cfg["downscaling"]["gdp_file"],
              county_sectors = cfg["downscaling"]["downscaled_sectors"],
              remove_fte_tech_mode = cfg["downscaling"]["remove_fte_tech_mode"],
              output_path = cfg["filepaths"]["downscaled_model_path"])

## (3) Integrating county data and enhancing the model ##

With the basic county-resolved model derived based on the downscaling of the national model, this step integrates county-level and other data for the analysis (and in general would allow for enhancement and integration of other data, scenario-related or other changes). For this analysis, this steps integrates the pre-defined *cooking* enhancements to the model. This mainly includes integration of:
- Scenario data describing the national cooking transitions of the five scenarios explored for the KNeCS (and discussed in <a href="https://www.energy.go.ke/sites/default/files/KAWI/Publication/KNeCS%20Strategy%20Modelling%20Report_Draft%204.pdf">its modelling report</a>), including the scenario adopted for the strategy
- Baseline data for current cookstove use across counties from a KNBS household survey


In [ ]:
# Process the model (this again makes use of the filepaths and parameters set out in the configuration file)
dp.process_county_model(input_path = cfg["filepaths"]["downscaled_model_path"],
                        dataconfig_file = cfg["filepaths"]["data_config_file_NM"],
                        datasets = cfg["county_processing"]["datasets"],
                        output_path = cfg["filepaths"]["county_processed_path"]
                        )

## (4) Run model ##

This step (split into 3 code cells) is running the model for the selected scenarios and saves the results.

The first cell below defines the analysis name (*KNeCS*) and run ID (*v004*). Do not change those for the workshop (as pre-run results will be used as running the model would require too much time).

In [ ]:
name = "KNeCS" # do not change for the workshop
run_id = "v004" # do not change for the workshop

The following cell loads the analysis-specific configuration file (the file can be found <a href="config_files/analysis_config_KNeCS.yaml">here</a>) that defines, among others:
- the scenarios to be run
- the spatial configuration of the runs

In [ ]:
with open("config_files/analysis_config_"+name+".yaml") as stream:
    try:
        acfg = yaml.safe_load(stream)
    except yaml.YAMLError as exc:
        print(exc)

The following cell defines the scenarios to be run (to overwrite the configuration file for ease of use of the notebook).

<b style='color:red;'>[Change required]:</b> Choose three scenarios out of the 5 scenarios modelled for the KNeCS and change the code cell below accordingly. Scenarios analysed for the KNeCS include the Business-As-Usual scenario (*BAU*), the Stated Policies scenario (*StatedPolicies*), the ecooking adoption scenario as adopted for the KNeCS (*KNeCS*), a simulated net zero scenario (*SimNetZero*), and an optimized net zero scenario (*OptNetZero*).

It is recommended to choose the BAU, KNeCS and one additonal scenario. For example, if the simulated net zero scenario is chosen as the third scenario, the cell below should contain the following

> scenarios = ["BAU","KNeCS","SimNetZero"]



In [ ]:
# Define the scenarios to be run
scenarios = ["BAU","KNeCS","SimNetZero"] # Change the text between the quotation marks to change the selected scenarios

The following cell actually runs the required scenarios. If the model run has already be performed before and results exist, the code will automatically skip this (computationally expensive) step.

In [ ]:
# Overwrite scenarios in the configuration based on input in the cell above
acfg["runs"]["scenarios"] = scenarios

# Run the model for each of the selected scenarios and save the results (this again makes use of the filepaths and parameters set out in the configuration file)
rp.run_model(dataconfig_file = cfg["filepaths"]["data_config_file_CW"],
              input_path = cfg["filepaths"]["county_processed_path"],
              model_file_path= acfg["runs"]["model_file"],
              scenario_list = acfg["scenarios"],
              spatial_config = acfg["runs"]["spatial_config"],
              output_path = cfg["filepaths"]["results_path"]+name+"/"+run_id,
              rename_set = {"COMMODITY":"FUEL"},
              glpk_dir = cfg["filepaths"]["glpk_dir"],
              agg_years = acfg["runs"]["agg_years"],
              agg_config = acfg["runs"]["agg_cfg"],
              agg_timeslices = acfg["runs"]["agg_ts"],
              solve = "optimize")

## (5) Analyse results ##

This last step – again split into a number of code cells, is about analysing the results. In particular, it creates a few graphs.

The first cell focuses on national aggregated results. It creates a visualization of the cooking pathways for the selected scenarios.

In [ ]:
# Plot overview of total, urban, and rural use of cookstoves
rp.plot_national(input_path=cfg["filepaths"]["results_path"]+name+"/"+run_id,
                  dataconfig_file=cfg["filepaths"]["data_config_file_CW"],
                  years_agg_file=acfg["runs"]["agg_years"],
                  scenario_list=acfg["runs"]["scenarios"],
                  naming=acfg["runs"]["naming"])

The following three cells are concered with two visualizations that show the variation of those pathways across counties. This is currently based on a relatively simply allocation of national targets across counties, mainly based on their urban/rural split and baseline use of stoves.

The first graph shows the fraction of cooking demand met by 5 key technologies in the form of a box plot, with focus counties highlighted.

The first graph shows the additional annual and peak electricity demand from e-cooking in the form of a box plot, with focus counties again highlighted.

<b style='color:red;'>[Change required]:</b> The first cell below can be used to choose up to 4 counties to highlight in the plot.

In [ ]:
counties = ["Baringo","Nairobi"]

In [ ]:
rp.plot_counties(input_path=cfg["filepaths"]["results_path"]+name+"/"+run_id,
                  dataconfig_file=cfg["filepaths"]["data_config_file_CW"],
                  years_agg_file=acfg["runs"]["agg_years"],
                  scenario_list=acfg["runs"]["scenarios"],
                  counties = counties,
                  list_counties = cfg["downscaling"]["list_counties_file"],
                  naming=acfg["runs"]["naming"])

In [ ]:
rp.plot_counties_impact(input_path=cfg["filepaths"]["results_path"]+name+"/"+run_id,
                  dataconfig_file=cfg["filepaths"]["data_config_file_CW"],
                  years_agg_file=acfg["runs"]["agg_years"],
                  scenario_list=acfg["runs"]["scenarios"],
                  counties = counties,
                  list_counties = cfg["downscaling"]["list_counties_file"],
                  naming=acfg["runs"]["naming"])